In [9]:
from pathlib import Path
import pandas as pd
import re

base_path = Path('C:/Users/Marcus/Documents/DSAI/Azure_WBS')
source = base_path / 'bronze' / 'new'

files = [str(p.name) for p in source.glob('*.csv')]
if not files:
    raise FileNotFoundError(f'No CSV files found in {source}')

dfs = {}
for file in files:
    file_path = source / file
    base_name = file_path.stem
    df_name = base_name.split('_')[0] if '_' in base_name else base_name

    print('Lade:', file, '->', df_name)
    dfs[df_name] = pd.read_csv(file_path)

for name, df in dfs.items():
    print(name, df.shape)


Lade: flow_schwartau.csv -> flow
Lade: humidity_schwartau.csv -> humidity
Lade: temperature_schwartau.csv -> temperature
Lade: weight_schwartau.csv -> weight
flow (2513836, 2)
humidity (1761, 2)
temperature (253430, 2)
weight (1761, 2)


In [10]:

# 1. standardize column names
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    return df

# 2. convert datetime-like columns to UTC
def unify_datetime(df, ambiguous_mode="infer"):
    df = df.copy()
    for col in df.columns:
        if "time" in col or "date" in col:
            s = pd.to_datetime(df[col], errors="coerce")

            if isinstance(s.dtype, pd.DatetimeTZDtype):
                df[col] = s.dt.tz_convert("UTC")
            else:
                ambiguous = ambiguous_mode
                if ambiguous_mode == "summer":
                    ambiguous = pd.Series(True, index=s.index)

                df[col] = s.dt.tz_localize(
                    "Europe/Berlin",
                    ambiguous=ambiguous,
                    nonexistent="shift_forward"
                ).dt.tz_convert("UTC")
    return df

# 3. fix data types
def fix_dtypes(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            num = pd.to_numeric(df[col], errors="ignore")
            if num.dtype != "object":
                df[col] = num

        if df[col].dtype == "object":
            lower = df[col].astype(str).str.lower()
            if set(lower.unique()) <= {"true", "false", "yes", "no", "1", "0", "nan"}:
                df[col] = lower.map({
                    "true": True, "yes": True, "1": True,
                    "false": False, "no": False, "0": False
                })
    return df

# 4. dataset-specific value corrections (matching sample notebook data-changing steps)
def apply_value_fixes(df, name):
    df = df.copy()

    if name == "humidity" and "humidity" in df.columns:
        df["humidity"] = df["humidity"].abs()

    if name == "weight" and "weight" in df.columns:
        df["weight"] = df["weight"].abs()
        df["weight"] = df["weight"].fillna(df["weight"].mean())

    return df

# 5. remove duplicates
def remove_duplicates(df):
    return df.drop_duplicates().reset_index(drop=True)

# 6. remove outliers (z-score)
def remove_outliers(df, z_thresh=3):
    df = df.copy()
    numeric_cols = df.select_dtypes(include="number").columns
    if len(numeric_cols) == 0:
        return df

    z = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std().replace(0, pd.NA)
    mask = ((z.abs() <= z_thresh) | z.isna()).all(axis=1)
    return df.loc[mask].reset_index(drop=True)

# 7. flow processing aligned with sample notebook
def process_flow(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)
    split_idx = df.shape[0] // 2

    departures = df.iloc[:split_idx].copy()
    arrivals = df.iloc[split_idx:].copy()

    flow = departures.merge(arrivals, on="timestamp", how="outer", suffixes=["_out", "_in"])
    flow = flow.sort_values("timestamp")

    # regularize to 1-minute timeline
    start_dt = flow["timestamp"].min()
    end_dt = flow["timestamp"].max()
    one_minute_index = pd.DataFrame({
        "timestamp": pd.date_range(start=start_dt, end=end_dt, freq="1min")
    })
    flow = flow.merge(one_minute_index, on="timestamp", how="right")

    return flow

# 8. LOOP over all DataFrames and build silver outputs
dfs_silver = {}

for name, df in dfs.items():
    df2 = clean_columns(df)

    if name in {"humidity", "weight"}:
        df2 = unify_datetime(df2, ambiguous_mode="summer")
    else:
        df2 = unify_datetime(df2, ambiguous_mode="infer")

    df2 = fix_dtypes(df2)
    df2 = apply_value_fixes(df2, name)

    if name == "flow":
        df2 = process_flow(df2)

    df2 = remove_duplicates(df2)
    df2 = remove_outliers(df2)

    dfs_silver[name] = df2




# Data writes

In [ ]:
from pathlib import Path
import re

# write all cleaned silver dataframes in a loop
sink = base_path / 'silver'
timestamp = pd.Timestamp.now().strftime('%Y-%m-%dT%H-%M-%S')

# derive source/location token (e.g., schwartau) from original filenames
known_streams = set(dfs_silver.keys())
source_name = 'unknown'
for file in files:
    stem = Path(file).stem
    tokens = [t for t in re.split(r'[_\\-]+', stem) if t]
    candidates = [t for t in tokens if t.lower() not in known_streams and not t.isdigit()]
    if candidates:
        source_name = candidates[0]
        break

for name, df in dfs_silver.items():
    out_dir = sink / name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f'{source_name}__{timestamp}.parquet'
    df.to_parquet(out_file, index=False)
    print(f'Wrote {name}: {out_file}')

Wrote flow: C:\Users\Marcus\Documents\DSAI\Azure_WBS\silver\flow\schwartau__2026-04-21T10-55-44.parquet
Wrote humidity: C:\Users\Marcus\Documents\DSAI\Azure_WBS\silver\humidity\schwartau__2026-04-21T10-55-44.parquet
Wrote temperature: C:\Users\Marcus\Documents\DSAI\Azure_WBS\silver\temperature\schwartau__2026-04-21T10-55-44.parquet
Wrote weight: C:\Users\Marcus\Documents\DSAI\Azure_WBS\silver\weight\schwartau__2026-04-21T10-55-44.parquet
